
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Lecture - Agent Bricks for Retrieval and Context Engineering
## Overview
This lecture introduces the core ideas behind retrieval agents and context engineering on Databricks. We'll cover why prompting alone isn't enough, how Retrieval Augmented Generation (RAG) fills the gap, what context engineering means for agent systems, and how Databricks Knowledge Assistants offer a managed path to production-grade RAG.

## Learning Objectives
By the end of this lecture, you will be able to:
1. Explain what retrieval agents are and why they are needed
2. Define context engineering and distinguish it from prompt engineering
3. Describe Databricks Knowledge Assistants as a managed retrieval agent solution
4. Identify the key components of a Knowledge Assistant

## A. Introduction to Retrieval Agents

### A1. Why Retrieval Agents?

Large Language Models (LLMs) are powerful, but they have hard limits. No amount of prompt refinement can overcome:

- **Knowledge cutoffs** - the model doesn't know about events after its training date
- **Hallucination** - the model fabricates plausible-sounding facts when it lacks real data
- **Missing private context** - the model has no access to your organization's proprietary documents

**Retrieval Augmented Generation (RAG)** addresses these limits by injecting relevant external data into the model's context at query time. Instead of relying on frozen training data, a RAG system **retrieves** relevant documents, **augments** the prompt with them, and lets the model **generate** a grounded response.

A **retrieval agent** is a concrete implementation of this pattern that handles query routing, retrieval orchestration, and context assembly within a real system.

<div style="max-width: 900px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 680 260" role="img" style="font-family: sans-serif;">
  <title>RAG Pipeline</title>
  <desc>User query flows through Retrieve, Augment, and Generate stages to produce a grounded response.</desc>
  <defs>
    <marker id="rag-arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#1B3139" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  </defs>

  <!-- User Query -->
  <rect x="15" y="90" width="120" height="60" rx="8" fill="#ffffff" stroke="#1B3139" stroke-width="1.5"/>
  <text x="75" y="124" text-anchor="middle" font-size="18" font-weight="600" fill="#0b2026">User Query</text>

  <!-- Arrow 1 -->
  <path d="M135 120 L175 120" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#rag-arrow)"/>

  <!-- Retrieve -->
  <rect x="180" y="90" width="130" height="60" rx="8" fill="#2272B4" fill-opacity="0.12" stroke="#2272B4" stroke-width="1.5"/>
  <text x="245" y="116" text-anchor="middle" font-size="18" font-weight="600" fill="#0b2026">Retrieve</text>
  <text x="245" y="134" text-anchor="middle" font-size="15" fill="#618794">AI Search</text>

  <!-- Arrow 2 -->
  <path d="M310 120 L350 120" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#rag-arrow)"/>

  <!-- Augment -->
  <rect x="355" y="90" width="130" height="60" rx="8" fill="#00A972" fill-opacity="0.12" stroke="#00A972" stroke-width="1.5"/>
  <text x="420" y="116" text-anchor="middle" font-size="18" font-weight="600" fill="#0b2026">Augment</text>
  <text x="420" y="134" text-anchor="middle" font-size="15" fill="#618794">Inject context</text>

  <!-- Arrow 3 -->
  <path d="M485 120 L525 120" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#rag-arrow)"/>

  <!-- Generate -->
  <rect x="530" y="90" width="130" height="60" rx="8" fill="#FF5F46" fill-opacity="0.12" stroke="#FF5F46" stroke-width="1.5"/>
  <text x="595" y="116" text-anchor="middle" font-size="18" font-weight="600" fill="#0b2026">Generate</text>
  <text x="595" y="134" text-anchor="middle" font-size="15" fill="#618794">LLM response</text>

  <!-- Knowledge Base below Retrieve -->
  <rect x="180" y="180" width="130" height="50" rx="8" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.2"/>
  <text x="245" y="204" text-anchor="middle" font-size="16" fill="#0b2026">Knowledge Base</text>
  <text x="245" y="220" text-anchor="middle" font-size="15" fill="#618794">Documents, PDFs</text>
  <path d="M245 180 L245 150" fill="none" stroke="#1B3139" stroke-width="1.5" marker-end="url(#rag-arrow)"/>
</svg>
</div>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<strong style="color: #0d47a1;">Learn More</strong>
<p style="margin: 8px 0 0 0; color: #333;">
See the Databricks documentation on Retrieval Augmented Generation (<a href="https://docs.databricks.com/aws/en/generative-ai/retrieval-augmented-generation#what-is-retrieval-augmented-generation" style="color: #1976d2;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/generative-ai/retrieval-augmented-generation#what-is-retrieval-augmented-generation" style="color: #1976d2;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/generative-ai/retrieval-augmented-generation#what-is-retrieval-augmented-generation" style="color: #1976d2;">GCP</a>) for a deeper dive.
</p>
</div>

## B. Introduction to Context Engineering


### B1. From Prompt Engineering to Context Engineering

**Prompt engineering** focuses on crafting the instruction text sent to a model. It involves choosing the right words, formatting, and examples. It's tactical and operates at the level of a single query.

**Context engineering** is a broader discipline. It's the art and science of designing the *entire information environment* that a model receives: not just the question, but all the supporting data, instructions, history, and constraints that shape the model's decision at inference time.

Think of it this way: prompt engineering is writing a good question on an exam. Context engineering is designing the entire exam room, including the reference materials on the desk, the instructions on the board, and the rules about what resources are allowed.

<div style="display: flex; align-items: center; gap: 20px; flex-wrap: wrap; justify-content: center; margin: 16px 0;">
  <!-- Prompt Engineering -->
  <div style="border: 1.5px solid #618794; border-radius: 12px; padding: 20px 24px; width: 260px; background: #ffffff;">
    <div style="font-size: 19px; font-weight: 700; color: #0b2026; text-align: center;">Prompt Engineering</div>
    <div style="font-size: 16px; color: #618794; text-align: center; margin-top: 6px;">Context window: the instruction text</div>
    <div style="border: 1px solid #618794; border-radius: 6px; background: #ffffff; padding: 6px 10px; text-align: center; margin-top: 14px; font-size: 15px; color: #0b2026;">System instructions</div>
    <div style="border: 1px solid #618794; border-radius: 6px; background: #ffffff; padding: 6px 10px; text-align: center; margin-top: 4px; font-size: 15px; color: #0b2026;">Few-shot examples</div>
    <div style="border: 1px dashed #2272B4; border-radius: 6px; background: rgba(34,114,180,0.08); padding: 4px 10px; text-align: center; margin-top: 4px; font-size: 15px; color: #0b2026;">User prompt</div>
  </div>
  <!-- Arrow -->
  <div style="display: flex; flex-direction: column; align-items: center; gap: 8px; min-width: 140px;">
    <div style="font-size: 15px; color: #618794; text-align: center; line-height: 1.4;">Better management<br/>for entire<br/>context state</div>
    <svg width="120" height="20" viewBox="0 0 120 20" style="display: block;">
      <defs>
        <marker id="ce-arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
          <path d="M2 1L8 5L2 9" fill="none" stroke="#1B3139" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
        </marker>
      </defs>
      <path d="M5 10 L110 10" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#ce-arrow)"/>
    </svg>
  </div>
  <!-- Context Engineering -->
  <div style="border: 2px solid #FF5F46; border-radius: 12px; padding: 20px 24px; width: 280px; background: #F9F7F4;">
    <div style="font-size: 19px; font-weight: 700; color: #FF5F46; text-align: center;">Context Engineering</div>
    <div style="font-size: 16px; color: #618794; text-align: center; margin-top: 4px;">Context window: the entire input environment</div>
    <div style="border: 1px solid #618794; border-radius: 6px; background: #ffffff; padding: 6px 10px; text-align: center; margin-top: 10px; font-size: 15px; color: #0b2026;">System instructions</div>
    <div style="border: 1px solid #618794; border-radius: 6px; background: #ffffff; padding: 6px 10px; text-align: center; margin-top: 4px; font-size: 15px; color: #0b2026;">Retrieved documents + metadata</div>
    <div style="border: 1px solid #618794; border-radius: 6px; background: #ffffff; padding: 6px 10px; text-align: center; margin-top: 4px; font-size: 15px; color: #0b2026;">Conversation history + user constraints (Lakebase)</div>
    <div style="border: 1px solid #618794; border-radius: 6px; background: #ffffff; padding: 6px 10px; text-align: center; margin-top: 4px; font-size: 15px; color: #0b2026;">Tool usage (MCP + Genie)</div>
    <div style="border: 1px dashed #2272B4; border-radius: 6px; background: rgba(34,114,180,0.08); padding: 4px 10px; text-align: center; margin-top: 6px; font-size: 15px; color: #0b2026;">User prompt</div>
  </div>
</div>

### B2. Why Context Engineering Matters for Agents

When building AI agents (systems that reason, plan, and take actions), context engineering becomes essential. An agent doesn't just answer one question; it makes a *sequence* of decisions, each of which depends on having the right information available.

Effective context engineering for agents involves:

- **Providing the right information at the right time** - supplying relevant retrieved data, tool outputs, and prior decisions without overwhelming the model with noise
- **Writing clear instructions that persist** - system prompts that define behavior, constraints, and output format across multiple turns
- **Managing state across turns** - summarizing conversation history, pruning irrelevant context, and keeping token budgets under control
- **Keeping the context trustworthy** - grounding the model strictly in retrieved facts and preventing hallucination through explicit instructions

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<strong style="color: #0d47a1;">Summary</strong>
<p style="margin: 8px 0 0 0; color: #333;">
The quality of an agent's output is bounded by the quality of its context. No model can reason well over bad inputs. Context engineering is the practice of making sure every token in the input window is earning its place.
</p>
</div>


## C. Overview of Agent Bricks Knowledge Assistant

<div style="display: flex; align-items: flex-start; gap: 10px; flex-wrap: nowrap; justify-content: center; margin: 16px 0; overflow-x: auto;">

  <!-- Box 1: Indexing -->
  <div style="border: 1.5px solid #618794; border-radius: 12px; padding: 16px; width: 200px; flex-shrink: 0; background: #ffffff;">
    <div style="font-size: 18px; font-weight: 700; color: #0b2026; text-align: center;">Indexing</div>
    <div style="border: 1px solid #618794; border-radius: 6px; background: #ffffff; padding: 8px 10px; margin-top: 14px;">
      <div style="font-size: 14px; color: #0b2026; font-weight: 600;">Ingest &amp; Parse</div>
      <div style="font-size: 12px; color: #618794; margin-top: 2px; line-height: 1.35;">UC volumes/tables → chunks + metadata</div>
    </div>
    <div style="border: 1px solid #618794; border-radius: 6px; background: #ffffff; padding: 8px 10px; margin-top: 6px;">
      <div style="font-size: 14px; color: #0b2026; font-weight: 600;">Embed &amp; Index</div>
      <div style="font-size: 12px; color: #618794; margin-top: 2px; line-height: 1.35;">databricks-gte-large-en → AI Search index</div>
    </div>
  </div>

  <!-- Arrow -->
  <div style="padding-top: 50px; flex-shrink: 0;">
    <div style="font-size: 22px; color: #1B3139;">&#10132;</div>
  </div>

  <!-- Box 2: Retrieval -->
  <div style="border: 1.5px solid #2272B4; border-radius: 12px; padding: 16px; width: 200px; flex-shrink: 0; background: #ffffff;">
    <div style="font-size: 18px; font-weight: 700; color: #0b2026; text-align: center;">Retrieval</div>
    <div style="border: 1px solid #618794; border-radius: 6px; background: #ffffff; padding: 8px 10px; margin-top: 14px;">
      <div style="font-size: 14px; color: #0b2026; font-weight: 600;">Receive Query</div>
      <div style="font-size: 12px; color: #618794; margin-top: 2px; line-height: 1.35;">User question hits endpoint</div>
    </div>
    <div style="border: 1px solid #618794; border-radius: 6px; background: #ffffff; padding: 8px 10px; margin-top: 6px;">
      <div style="font-size: 11px; color: #618794; font-weight: 600; letter-spacing: 0.1em;">INSTRUCTED RETRIEVER</div>
      <div style="font-size: 14px; color: #0b2026; font-weight: 600;">Plan Retrieval</div>
      <div style="font-size: 12px; color: #618794; margin-top: 2px; line-height: 1.35;">Pick sources → tailored sub-queries</div>
    </div>
    <div style="border: 1px solid #618794; border-radius: 6px; background: #ffffff; padding: 8px 10px; margin-top: 6px;">
      <div style="font-size: 14px; color: #0b2026; font-weight: 600;">Search &amp; Rank</div>
      <div style="font-size: 12px; color: #618794; margin-top: 2px; line-height: 1.35;">AI Search → rerank → evidence set</div>
    </div>
  </div>

  <!-- Arrow -->
  <div style="padding-top: 50px; flex-shrink: 0;">
    <div style="font-size: 22px; color: #1B3139;">&#10132;</div>
  </div>

  <!-- Box 3: Generation -->
  <div style="border: 2px solid #FF5F46; border-radius: 12px; padding: 16px; width: 200px; flex-shrink: 0; background: #F9F7F4;">
    <div style="font-size: 18px; font-weight: 700; color: #FF5F46; text-align: center;">Generation</div>
    <div style="border: 1px solid #FF5F46; border-radius: 6px; background: #ffffff; padding: 10px; margin-top: 14px;">
      <div style="font-size: 14px; color: #0b2026; font-weight: 600;">Generate Answer</div>
      <div style="font-size: 12px; color: #618794; margin-top: 2px; line-height: 1.35;">LLM grounds on chunks → cited answer</div>
      <div style="font-size: 11px; color: #618794; margin-top: 6px; font-style: italic;">doc &amp; page-level citations</div>
    </div>
  </div>

  <!-- Arrow -->
  <div style="padding-top: 50px; flex-shrink: 0;">
    <div style="font-size: 22px; color: #1B3139;">&#10132;</div>
  </div>

  <!-- Box 4: Feedback Loop -->
  <div style="border: 1.5px dashed #618794; border-radius: 12px; padding: 16px; width: 200px; flex-shrink: 0; background: #ffffff;">
    <div style="font-size: 18px; font-weight: 700; color: #0b2026; text-align: center;">Feedback Loop</div>
    <div style="border: 1px solid #618794; border-radius: 6px; background: #ffffff; padding: 8px 10px; margin-top: 14px;">
      <div style="font-size: 11px; color: #618794; font-weight: 600; letter-spacing: 0.1em;">ALHF</div>
      <div style="font-size: 14px; color: #0b2026; font-weight: 600;">Log &amp; Learn</div>
      <div style="font-size: 12px; color: #618794; margin-top: 2px; line-height: 1.35;">Trace + SME feedback → tune KA</div>
      <div style="font-size: 11px; color: #FF5F46; margin-top: 6px; font-weight: 600;">refines retrieval &amp; generation</div>
    </div>
  </div>

</div>


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #4299E0); color: white; padding: 12px 18px; cursor: pointer; font-weight: 600; font-size: 12pt; border-radius: 8px; user-select: none;">
What are the four stages of a Knowledge Assistant pipeline?
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 16px 20px; background: #F9F7F4; font-size: 12pt; line-height: 1.7; color: #333;">

<p style="margin-top: 10px;">A Knowledge Assistant request flows through four stages: two offline (preparing the index) and two online (handling the request), with a feedback loop that tunes the system over time.</p>

<ol style="margin: 12px 0; padding-left: 24px;">
<li><strong>Pipelines/transformations</strong>: Ingest UC volumes/tables or existing VS indexes, then <strong>parse, chunk, embed, and index</strong> documents (plus normalize metadata).</li>
<li><strong>Retrieval</strong>: Receive the user query, <strong>plan retrieval</strong> with Instructed Retriever across the configured sources, then <strong>search and rank</strong> results from one or more vector indexes.</li>
<li><strong>Generation</strong>: Use an LLM to <strong>generate a grounded answer</strong> from the retrieved context, including <strong>doc/page-level citations</strong> back to the sources (citations are for users and evaluators, not only "testing").</li>
<li><strong>Feedback loop</strong>: Uses MLflow evaluation (tracing, LLM judges, task-specific eval datasets/guidelines) as the evaluation layer and <strong><a href="https://www.databricks.com/blog/agent-learning-human-feedback-alhf-databricks-knowledge-assistant-case-study" target="_blank">Agent Learning from Human Feedback (ALHF)</a></strong> to refine both retrieval behavior and answer generation over time that acts as the learning/optimization layer.</li>
</ol>

</div>
</details>


### C1. Knowledge Source Types

A Knowledge Assistant can draw from multiple knowledge sources simultaneously (up to 10 per agent):

| Source Type                      | Description                                                                                                                                           | Best For |
|---|---|---|
| **Files in UC Volume**          | Point to a Unity Catalog Volume containing `txt`, `pdf`, `md`, `ppt/pptx`, or `doc/docx` files. KA parses, chunks, embeds, and indexes them for you. Files >50 MB are automatically skipped. | Document collections that change over time (policies, manuals, specs) |
| **AI Search Index**         | Attach a pre-built AI Search index (must use `databricks-gte-large-en` as the embedding model). You own the parsing/chunking/indexing pipeline. | Custom RAG pipelines where you already manage the index and schema |
| **Files in UC Table / File Table** | Use a UC table (must be a **streaming table** or have **Change Data Feed enabled**) with a `content` column (BINARY/STRING) storing file contents plus a `_metadata`/`metadata` struct with file path/name/size/mod time. KA ingests those files and builds its own index. | Documents ingested via connectors (SharePoint, Google Drive, Jira, Confluence) that land as file tables |

### C2. Declarative vs. Code-First

Databricks offers two approaches to building retrieval agents:

|                       | Declarative (Knowledge Assistant)                           | Code-First (Custom Agents / Supervisor)                           |
|---|---|---|
| **Approach**          | Declare what you want; the system learns and optimizes how | Write retrieval logic, prompts, tools, and orchestration yourself |
| **Setup time**        | Minutes                                                     | Hours to days                                                     |
| **Customizability**   | Configuration- and feedback-driven                          | Essentially unlimited                                             |
| **Optimization**      | Automatic via ALHF feedback loop and research upgrades      | Manual tuning and experimentation                                 |
| **Best for**          | High-quality Q&A over docs, fast prototyping               | Novel architectures, complex multi-step / tool-heavy workflows   |

This course focuses on the **declarative** path. We’ll explore the components that power Knowledge Assistants (document parsing, chunking, and AI Search) so you understand the building blocks, then bring them together using the managed approach.

<div style="width: 100%; font-family: sans-serif;">

<div style="background: #F9F7F4; border-radius: 10px; padding: 24px 28px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); border-top: 6px solid #FF5F46;">

  <img src="./Includes/images/genie-code.png" style="height: 64px; margin-bottom: 10px;">

  <div style="font-size: 15pt; color: #0b2026; line-height: 1.7; margin-bottom: 16px;">
    Want to know more about how Agent Bricks integrates with other Databricks features? Ask Genie Code. Click on the genie icon <img src="./Includes/images/genie-icon.png" style="height: 32px; vertical-align: middle;"> and begin querying. For example, click the <strong>Copy</strong> button below and paste into <strong>Genie Code</strong>.
  </div>

  <div style="display: flex; align-items: center; gap: 10px; background: #fff; border: 1px solid #ddd; border-radius: 6px; padding: 10px 14px; font-size: 14pt; font-family: monospace; color: #0b2026;">
    <span id="genie-query" style="flex: 1;">How can Agent Bricks be used with Databricks Apps?</span>
    <button onclick="
      var text = document.getElementById('genie-query').innerText;
      var ta = document.createElement('textarea');
      ta.value = text;
      ta.style.position = 'fixed';
      ta.style.opacity = '0';
      document.body.appendChild(ta);
      ta.select();
      document.execCommand('copy');
      document.body.removeChild(ta);
      this.innerText = 'Copied!';
      var btn = this;
      setTimeout(function(){ btn.innerText = 'Copy'; }, 2000);
    " style="background: #FF5F46; color: white; border: none; border-radius: 4px; padding: 4px 12px; font-size: 13pt; cursor: pointer; white-space: nowrap;">Copy</button>
  </div>

</div>

</div>

## D. Conclusion

In this lecture, we established three key ideas:

1. **Retrieval agents** solve the fundamental limitations of LLMs (knowledge cutoffs, hallucination, missing private context) by injecting relevant data at query time through RAG.

2. **Context engineering** is the broader discipline of designing the entire input environment for a model, going beyond prompt crafting to manage retrieved data, system instructions, and conversation state.

3. **Knowledge Assistants** on Databricks provide a managed, declarative way to build retrieval agents that handles parsing, indexing, and serving automatically while you focus on the data and the outcome.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
